In [ ]:
import pandas as pd
from langchain_community.document_loaders import DataFrameLoader

In [ ]:
df = pd.read_excel("step3result copy 2.xlsx") 
df

In [ ]:
categories = df['category'].unique()
user_question = "Does a diagnosis of a connective tissue disease contribute to a post-dural spinal puncture headache?"

In [24]:
from langchain.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_openai import AzureChatOpenAI
import os
from langchain.chains.llm import LLMChain
from langchain.prompts import PromptTemplate
from langchain.chains.combine_documents.stuff import StuffDocumentsChain
from langchain.chains import MapReduceDocumentsChain, ReduceDocumentsChain
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import DataFrameLoader

# remove this later
CHAT = AzureChatOpenAI(
    azure_endpoint="https://nlp-ai-svc.openai.azure.com/",
    openai_api_version="2024-02-15-preview",
    azure_deployment="ChatGPT4128k",
    openai_api_type="azure",
    temperature=0,
    model_name="gpt-4",
    openai_api_key=os.environ.get("OPENAI_API_KEY")
)


# CATEGORIZATION_SUMMARY_SYSTEM_TEMPLATE = """I am considering a scoping reivew project for given category I want to understand how the existing literature guides. Write a single detailed summary based on the full texts of the article for each category."""

SUMMARIZE_CATEGORY_TEMPLATE = "I am working  on a scoping review to address this question: {question}\n\n Currently, I am summarizing articles by expert-defined categories. All of the article summaries below were assigned the category {category}. Write a single paragrph final summary of the following journal article summaries, focusing on my question. Cite each article in the paragraph in APA format. The article summaries are separated by '---'"

SUMMARIZE_ARTICLE_TEMPLATE = "I am working  on a scoping review to address this question: {question}\n\n Currently, I am summarizing articles by expert-defined categories. The article below was assigned the category {category}. Write a single paragrph summarizing the artcle, focusing on my question. The entire paragraph will be cited using only the article being summarized. Write the paragaraph so providing only the citation for the single article will be appropriate."



HUMAN_TEMPLATE = """
Content to summarize:
{content}
"""

category_system_message_prompt = SystemMessagePromptTemplate.from_template(SUMMARIZE_CATEGORY_TEMPLATE)
article_system_message_prompt = SystemMessagePromptTemplate.from_template(SUMMARIZE_ARTICLE_TEMPLATE)

human_message_prompt = HumanMessagePromptTemplate.from_template(HUMAN_TEMPLATE)

category_summary_chat_prompt = ChatPromptTemplate.from_messages([category_system_message_prompt, human_message_prompt])
article_summary_chat_prompt = ChatPromptTemplate.from_messages([article_system_message_prompt, human_message_prompt])


In [25]:
for current_category in categories:
    
    filtered_rows = df[(df['category'] == current_category) & (df['Text'] != "Text not available")]
    # if not filtered_rows.empty:
    #     new_df = pd.concat([new_df, filtered_rows])
    article_summaries = []
    for _, row in filtered_rows.iterrows():
        result = CHAT.invoke(article_summary_chat_prompt.format_prompt(
        question=user_question,
        category=current_category,
        content=row.Text,
        ).to_messages())
        # TODO: double check citation column name
        summary_to_keep = f"APA Citation: {row.APA_Citation}\n\n Summary: {result.content}\n\n --- "
        print(summary_to_keep)
        article_summaries.append(summary_to_keep)

    text_to_summarize = "\n\n".join(article_summaries)
   # text_to_summarize = f"Category of current articles: {current_category} \n\n" + text_to_summarize
    
    result = CHAT.invoke(categorization_summary_chat_prompt.format_prompt(
        question=user_question,
        category=current_category,
        articles=text_to_summarize
        ).to_messages())
    print(result.content)
    print("************")
    

APA Citation: Wijayanayaka, S., Guha, A., Sivanesan, K., & Veerasingham, M. (2021). Extra-axial haemorrhage in a patient with Alport syndrome after epidural anaesthesia. BMJ Case Reports CP, 14(6), e242160.

 Summary: In a case report by Wijayanayaka et al. (2021), an 18-year-old woman with Alport syndrome experienced a postpartum tonic-clonic seizure and was found to have an extra-axial hemorrhage (both extradural and subdural) following epidural anesthesia for a ventouse delivery. The patient initially presented with headaches and pain at the epidural site, which improved when moving from an erect to a supine position, suggesting a possible postdural puncture headache (PDPH). However, the subsequent seizure and imaging studies confirmed the presence of hemorrhage. The authors highlight the rarity of such complications, with intracranial subdural hematoma (ISH) following epidural anesthesia in obstetrics estimated at 1:500,000 to 1:250,000. The case underscores the importance of diffe

KeyboardInterrupt: 

In [23]:
# Map reduce
#Seems to get stuck

# Map
map_template = "I am working  on a scoping literature review to address this question: " + user_question + \
"\n\nCurrently, I am summarizing articles by expert-defined categories. All of the articles below were assigned the category " + current_category + \
"""{docs}
\n\nSummarize the provided articles, focusing on my question. Cite each article in your summary using APA format. \n\n""" 
map_prompt = PromptTemplate.from_template(map_template)
map_chain = LLMChain(llm=CHAT, prompt=map_prompt)

reduce_template = "I am working  on a scoping literature review to address this question: " + user_question + \
"\n\nCurrently, I am summarizing articles by expert-defined categories. All of the articles below were assigned the category " + current_category + \
"""The following is set of summaries of articles:
{docs}
Take these and distill it into a final, consolidated summary of the main themes. Maintain citations in the APA format."""
reduce_prompt = PromptTemplate.from_template(reduce_template)
reduce_chain = LLMChain(llm=CHAT, prompt=reduce_prompt)

# Takes a list of documents, combines them into a single string, and passes this to an LLMChain
combine_documents_chain = StuffDocumentsChain(
    llm_chain=reduce_chain, document_variable_name="docs"
)

# Combines and iteratively reduces the mapped documents
reduce_documents_chain = ReduceDocumentsChain(
    # This is final chain that is called.
    combine_documents_chain=combine_documents_chain,
    # If documents exceed context for `StuffDocumentsChain`
    collapse_documents_chain=combine_documents_chain,
    # The maximum number of tokens to group documents into.
    token_max=100000,
)

# Combining documents by mapping a chain over them, then combining results
map_reduce_chain = MapReduceDocumentsChain(
    # Map chain
    llm_chain=map_chain,
    # Reduce chain
    reduce_documents_chain=reduce_documents_chain,
    # The variable name in the llm_chain to put the documents in
    document_variable_name="docs",
    # Return the results of the map steps in the output
    return_intermediate_steps=False,
)



for current_category in categories:
    
    filtered_rows = df[(df['category'] == current_category) & (df['Text'] != "Text not available")]
    text_columns = df[['Text', 'APA_Citation']]
    loader = DataFrameLoader(text_columns, page_content_column="Text")
    docs = loader.load()
    # if not filtered_rows.empty:
    #     new_df = pd.concat([new_df, filtered_rows])
    text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=100000, chunk_overlap=0
    )
    split_docs = text_splitter.split_documents(docs)
    print(map_reduce_chain.run(split_docs))


KeyboardInterrupt: 

In [ ]:
filtered_rows[['category','Text', 'APA_Citation']].iloc[0].Text

In [ ]:
result.content